# Step 1 — Build the Data Foundation (Bronze → Silver → Gold)

**Mistplay Genie Workshop**

In this first step you generate a realistic e-commerce dataset and build a clean **medallion** pipeline:

- **Bronze (raw)** — synthetic landing data with realistic mess (string dates, inconsistent casing, a few NULLs and duplicate rows).
- **Silver** — cleaned, typed, deduplicated, conformed.
- **Gold** — a modeled **star schema** (dimensions + facts) with **rich table & column comments**. This is what your Genie space is built on in Step 2.

The domain: **orders, transactions, and shipments** across customers, products, and geographies (US states + Canadian provinces).

> All logic lives in the `workshop_lib` package next to this notebook — each cell just calls one function, so you can open and read the code any time.

## Before you run

Set the widgets at the top of this notebook:

1. **Catalog** — a catalog you can create schemas in (your own).
2. **Schema** — defaults to `mistplay_genie_workshop`.
3. **SQL Warehouse ID** — used in later Genie steps (leave blank for now if you like).
4. **Your email** — used to name your Genie space in later steps.

Then run the cells top to bottom.

In [0]:
import os, sys

# Make the workshop_lib package (in this notebook's folder) importable.
nb_dir = os.getcwd()
if nb_dir not in sys.path:
    sys.path.insert(0, nb_dir)

import importlib
import workshop_lib.config, workshop_lib.data_gen, workshop_lib.medallion
for m in (workshop_lib.config, workshop_lib.data_gen, workshop_lib.medallion):
    importlib.reload(m)

from workshop_lib.config import WorkshopConfig, setup_widgets
from workshop_lib import data_gen, medallion

print("workshop_lib loaded")

In [0]:
# Create the widgets, then read them into a validated config.
setup_widgets(dbutils)

cfg = WorkshopConfig.from_widgets(dbutils)
print("Config:", cfg.summary())

cfg.ensure_schema(spark)
print("Schema ready:", cfg.full_schema)

In [0]:
# Persist config so notebooks 02-07 can load it without widgets.
config_table = cfg.save_config(spark)
print(f"Config saved to {config_table} — notebooks 02-07 will read from here.")

## 1. Raw / Bronze layer

`data_gen.generate_raw` creates six raw landing tables (`bronze_*`):

`bronze_customers`, `bronze_products`, `bronze_geography`, `bronze_orders`, `bronze_transactions`, `bronze_shipments`.

They are **deliberately messy** — dates stored as strings, inconsistent casing (e.g. `fedex` vs `FedEx`), a few NULL emails, and some duplicate order lines — so the silver step has real cleaning to do. Everything is reproducible via the fixed `seed`.

In [0]:
counts = data_gen.generate_raw(spark, cfg, seed=42)
for t, n in counts.items():
    print(f"{n:>6}  {t}")

## 2. Silver layer

`medallion.build_silver` cleans and conforms bronze → `silver_*`:

- parse string dates to real `DATE` types,
- standardize inconsistent casing (segments, carriers) to canonical values,
- drop duplicate rows on each table's key.

In [0]:
counts = medallion.build_silver(spark, cfg)
for t, n in counts.items():
    print(f"{n:>6}  {t}")

## 3. Gold layer (+ comments)

`medallion.build_gold` models the clean **star schema** and applies **rich table & column comments** to every gold table:

- Dimensions: `dim_customers`, `dim_products`, `dim_geography`
- Facts: `fact_orders`, `fact_transactions`, `fact_shipments`

Note the **code-valued columns** kept intentionally as codes — `state_code` (`CA`=California), `country_code`, `payment_method`, `status`, `currency`. These are the raw material for the **Step 3 value-matching** lesson. The rich comments here also lift what Genie can do in Step 2 with no extra tuning.

In [0]:
counts = medallion.build_gold(spark, cfg)
for t, n in counts.items():
    print(f"{n:>6}  {t}")

## 4. Validate

Confirm the tables exist, peek at the gold data, and verify the comments landed (scroll to the column comments in the DESCRIBE output).

In [0]:
display(spark.sql(f"SHOW TABLES IN {cfg.full_schema}"))

In [0]:
display(spark.table(cfg.gold("fact_orders")).limit(10))

In [0]:
# Verify table + column comments were applied (Genie reads these in Step 2).
display(spark.sql(f"DESCRIBE TABLE EXTENDED {cfg.gold('fact_transactions')}"))

## Recap & what's next

You now have a **documented gold star schema** — six tables with rich comments describing every column, including deliberate code columns and business flags (`is_internal`, transaction `status`, unshipped orders).

**→ Step 2** creates a **bare Genie space** on a few gold tables and shows the two question sets: what it *can* answer already (thanks to your comments) and what it *can't* yet (value matching) — which motivates Step 3.